# Week 5: Baseline Models for Customer Churn Prediction

## Objectives
1. Load and explore the Telco Customer Churn dataset
2. Prepare data for machine learning
3. Train baseline models (Logistic Regression, Random Forest)
4. Evaluate and compare model performance
5. Visualize results with ROC curves

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

## 2. Data Loading

In [ ]:
# Load the dataset
# Option 1: Load from local file
# df = pd.read_csv('../data/Telco-Customer-Churn.csv')

# Option 2: Load from URL (recommended)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Basic info
print("Dataset Info:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Check target distribution
print(f"\nTarget distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].value_counts(normalize=True)['Yes']:.2%}")

In [ ]:
# Check data types and unique values
print("Data types:")
print(df.dtypes)
print(f"\nUnique values in each column:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

## 4. Data Preparation

In [ ]:
# Handle TotalCharges - convert to numeric and fill missing
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Missing TotalCharges after conversion: {df['TotalCharges'].isnull().sum()}")

# Fill missing TotalCharges with 0 (customers with 0 tenure)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop customerID (not useful for prediction)
df = df.drop('customerID', axis=1)

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Data preparation complete!")
df.head()

In [ ]:
# Identify feature types
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in df.columns if col not in numeric_features + ['Churn']]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Encode categorical variables
df_encoded = df.copy()

# Use Label Encoding for binary, One-Hot for multi-category
label_encoders = {}

for col in categorical_features:
    if df_encoded[col].nunique() == 2:
        # Label encoding for binary
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le
    else:
        # One-hot encoding for multi-category
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=True)
        df_encoded = pd.concat([df_encoded.drop(col, axis=1), dummies], axis=1)

print(f"Encoded dataset shape: {df_encoded.shape}")
df_encoded.head()

## 5. Train/Test Split

In [ ]:
# Prepare features and target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Stratified split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X):.1%})")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X):.1%})")
print(f"\nTraining set churn rate: {y_train.mean():.2%}")
print(f"Test set churn rate: {y_test.mean():.2%}")

In [ ]:
# Scale numeric features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Scale only numeric features
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print("Features scaled!")

## 6. Model Training

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Dictionary to store results
results = {}
trained_models = {}

print("Training models...\n")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Use scaled data for Logistic Regression, original for Random Forest
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    
    trained_models[name] = model
    print(f"  ✓ Training complete!")

print("\nAll models trained!")

## 7. Results Comparison

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("Model Comparison:")
print("=" * 70)
print(results_df.to_string())

# Save results
results_df.to_csv('../outputs/baseline_results.csv')
print("\nResults saved to ../outputs/baseline_results.csv")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot of metrics
results_df.plot(kind='bar', ax=axes[0])
axes[0].set_title('Model Performance Comparison')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=45)

# Heatmap
sns.heatmap(results_df, annot=True, cmap='YlOrRd', fmt='.3f', ax=axes[1])
axes[1].set_title('Performance Metrics Heatmap')

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. ROC Curve Visualization

In [ ]:
# Plot ROC curves
plt.figure(figsize=(10, 8))

for name, model in trained_models.items():
    # Get predictions
    if name == 'Logistic Regression':
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

# Plot diagonal
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)

plt.savefig('../outputs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (name, model) in enumerate(trained_models.items()):
    # Get predictions
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Plot
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name}\nConfusion Matrix')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xticklabels(['No Churn', 'Churn'])
    axes[idx].set_yticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Detailed Classification Reports

In [ ]:
# Print detailed classification reports
for name, model in trained_models.items():
    print(f"\n{'='*50}")
    print(f"Classification Report: {name}")
    print(f"{'='*50}")
    
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## 11. Feature Importance (Random Forest)

In [ ]:
# Get feature importance from Random Forest
rf_model = trained_models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), y='feature', x='importance')
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=14)
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

## 12. Summary and Conclusions

In [ ]:
# Summary
print("="*70)
print("WEEK 5 SUMMARY: BASELINE MODELS")
print("="*70)

print("\n📊 Dataset:")
print(f"   • Total samples: {len(df):,}")
print(f"   • Features: {X.shape[1]}")
print(f"   • Churn rate: {y.mean():.1%}")

print("\n🤖 Models Trained:")
for name in models.keys():
    print(f"   • {name}")

print("\n📈 Best Performance (ROC-AUC):")
best_model = results_df['roc_auc'].idxmax()
best_score = results_df.loc[best_model, 'roc_auc']
print(f"   • {best_model}: {best_score:.4f}")

print("\n📋 Model Comparison:")
print(results_df.to_string())

print("\n💡 Key Insights:")
print("   • Random Forest handles non-linear relationships well")
print("   • Logistic Regression provides interpretable coefficients")
print("   • Contract type and tenure are strong predictors")
print("   • Class imbalance affects precision-recall trade-off")

print("\n🚀 Next Steps:")
print("   • Build ML Pipelines (Week 6)")
print("   • Hyperparameter tuning with GridSearchCV")
print("   • Feature engineering and selection")
print("   • Address class imbalance")

print("\n" + "="*70)

## Exercises

1. **Try different random seeds** - How stable are the results across different splits?
2. **Experiment with class weights** - Add `class_weight='balanced'` to handle imbalance
3. **Try XGBoost** - Install xgboost and compare performance
4. **Threshold tuning** - Find the optimal threshold for F1 score
5. **Cross-validation** - Use StratifiedKFold for more robust evaluation